In [3]:
# ===============================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ===============================
import os
import glob
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")

PASTA_TREINO = "DADOS DE TREINAMENTO"
PASTA_ENTRADA = "ENTRADA"

print("Bibliotecas importadas com sucesso.")


# ============================================================
# 2. PARÂMETROS DO MODELO (solicitados via campo de texto)
# ============================================================
def perguntar(texto, padrao, tipo=str):
    '''Pergunta um valor ao usuário via campo de texto (input),
    retornando o valor padrão caso o campo seja deixado em branco.'''
    resposta = input(f"{texto} [padrão = {padrao}]: ").strip()
    if resposta == "":
        return padrao
    if tipo == int:
        return int(resposta)
    if tipo == float:
        return float(resposta)
    if tipo == bool:
        return resposta.lower() in ("s", "sim", "y", "yes", "true", "1")
    return resposta


def perguntar_opcao(texto, opcoes, padrao_idx=0):
    '''Apresenta uma lista numerada de opções e retorna a opção escolhida pelo usuário.
    Repete a pergunta até que uma escolha válida seja informada, evitando erros de digitação.'''
    print(f"\n{texto}")
    for i, opcao in enumerate(opcoes, start=1):
        marcador = " (padrão)" if (i - 1) == padrao_idx else ""
        print(f"  {i}. {opcao}{marcador}")
    while True:
        resposta = input(f"Escolha uma opção [1-{len(opcoes)}] (Enter = padrão): ").strip()
        if resposta == "":
            return opcoes[padrao_idx]
        if resposta.isdigit():
            indice = int(resposta)
            if 1 <= indice <= len(opcoes):
                return opcoes[indice - 1]
        print(f"Opção inválida. Digite um número entre 1 e {len(opcoes)}.")


print("\n=== Parâmetros da Regressão Logística ===\n")

# --------------------------------------------------------------------------
# Os parâmetros abaixo deixaram de ser perguntados ao usuário e passaram a
# usar sempre os valores padrão sugeridos:
#   - penalty       = "l2"
#   - C              = 1.0
#   - class_weight   = None (sem tratamento de classes desbalanceadas)
#   - fit_intercept  = True (inclui o termo independente na equação)
# --------------------------------------------------------------------------
penalty_escolhida = "l2"
penalty = penalty_escolhida

# O solver precisa ser compatível com o tipo de regularização escolhido; escolhemos
# automaticamente o solver adequado para evitar erros de incompatibilidade do scikit-learn.
mapa_solver = {"l2": "lbfgs", "l1": "liblinear", "elasticnet": "saga", "none": "lbfgs"}
solver = mapa_solver[penalty_escolhida]

l1_ratio = None  # só seria usado caso penalty fosse "elasticnet"

C = 1.0

class_weight = None

fit_intercept = True

max_iter = perguntar("Número máximo de iterações para convergência (max_iter)", 1000, int)

random_state = 42

test_size = perguntar(
    "Proporção dos dados que será separada para VALIDAÇÃO/TESTE, e não usada no treino "
    "(ex.: 0.2 = 20% para teste e 80% para treino; use 0 para treinar com 100% dos dados)",
    0.2, float
)

print("\n=== Parâmetros definidos ===")
print(f"penalty         = {penalty} (padrão)")
print(f"solver          = {solver} (escolhido automaticamente conforme o penalty)")
print(f"C               = {C} (padrão)")
print(f"class_weight    = {class_weight} (padrão, sem tratamento de desbalanceamento)")
print(f"fit_intercept   = {fit_intercept} (padrão, intercepto incluído)")
print(f"max_iter        = {max_iter}")
print(f"random_state    = {random_state}")
print(f"test_size       = {test_size}")


# =============================================
# 3. CARREGAMENTO DOS DADOS DE TREINAMENTO
# =============================================
def localizar_planilha(pasta):
    caminho_pasta = os.path.join(os.getcwd(), pasta)
    if not os.path.isdir(caminho_pasta):
        raise FileNotFoundError(
            f"A pasta '{pasta}' não foi encontrada em: {caminho_pasta}\n"
            f"Crie essa pasta no mesmo local do notebook e coloque o arquivo .xlsx dentro dela."
        )
    arquivos = glob.glob(os.path.join(caminho_pasta, "*.xlsx")) + glob.glob(os.path.join(caminho_pasta, "*.xls"))
    if len(arquivos) == 0:
        raise FileNotFoundError(f"Nenhum arquivo .xlsx/.xls encontrado na pasta '{caminho_pasta}'.")
    if len(arquivos) > 1:
        print(f"Aviso: mais de um arquivo encontrado em '{caminho_pasta}'. Usando o primeiro: {arquivos[0]}")
    return arquivos[0]


print("\n=== Carregando dados de treinamento ===")
caminho_treino = localizar_planilha(PASTA_TREINO)
print(f"Arquivo de treinamento encontrado: {caminho_treino}")

df_treino = pd.read_excel(caminho_treino)

if "Classe" not in df_treino.columns:
    raise ValueError("A planilha de treinamento precisa conter uma coluna chamada 'Classe'.")

print(f"Dimensões dos dados: {df_treino.shape[0]} linhas x {df_treino.shape[1]} colunas")
display(df_treino.head())


# =====================================================================
# 4. PRÉ-PROCESSAMENTO
# =====================================================================
X = df_treino.drop(columns=["Classe"]).copy()
y = df_treino["Classe"].copy()

colunas_originais = list(X.columns)

# Separa quais colunas são categóricas (texto) e quais já são numéricas.
# Usamos pd.api.types.is_numeric_dtype em vez de comparar com "object" diretamente,
# pois versões mais novas do pandas podem usar o tipo "string" para colunas de texto.
colunas_categoricas = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
colunas_numericas = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]

print("\nColunas categóricas identificadas:", colunas_categoricas if colunas_categoricas else "(nenhuma)")
print("Colunas numéricas identificadas:", colunas_numericas if colunas_numericas else "(nenhuma)")

# Tratamento de valores ausentes simples
medianas_treino = {}
for col in colunas_numericas:
    mediana = X[col].median()
    medianas_treino[col] = mediana
    X[col] = X[col].fillna(mediana)
for col in colunas_categoricas:
    X[col] = X[col].fillna("Desconhecido")

# -----------------------------------------------------------------------
# CODIFICAÇÃO DE VARIÁVEIS CATEGÓRICAS: ONE-HOT ENCODING
# -----------------------------------------------------------------------
# Diferente do Label Encoding (que atribui números 0,1,2... e cria uma ordem
# artificial entre as categorias), o One-Hot Encoding cria uma coluna binária
# (0/1) para cada categoria. Isso é o mais adequado aqui, já que não existe
# relação de ordem/importância entre os valores possíveis dos atributos.
#
# drop_first=True remove uma categoria de cada atributo (a "categoria de
# referência"), evitando redundância entre as colunas (dummy variable trap).
# O efeito das demais categorias é sempre lido EM RELAÇÃO a essa categoria
# de referência.
print("\n" + "=" * 70)
print("TRANSFORMAÇÃO: ONE-HOT ENCODING DOS ATRIBUTOS CATEGÓRICOS")
print("=" * 70)

if colunas_categoricas:
    print("\nTabela ANTES da transformação (valores originais):")
    display(X[colunas_categoricas].head())

    for col in colunas_categoricas:
        categorias = sorted(X[col].unique())
        print(f"\nAtributo '{col}': categorias encontradas = {categorias}")

    X_codificado = pd.get_dummies(X, columns=colunas_categoricas, drop_first=True)

    # Categoria de referência (a que foi removida em cada atributo) = a que
    # NÃO aparece em nenhum nome de coluna gerado para aquele atributo.
    print("\nCategoria de referência (baseline) removida em cada atributo:")
    for col in colunas_categoricas:
        categorias = sorted(X[col].unique())
        colunas_geradas = [c for c in X_codificado.columns if c.startswith(f"{col}_")]
        categorias_presentes = [c[len(col) + 1:] for c in colunas_geradas]
        referencia = [c for c in categorias if c not in categorias_presentes][0]
        print(f"  '{col}': referência = '{referencia}'  →  colunas geradas: {colunas_geradas}")

    print("\nTabela DEPOIS da transformação (0/1 por categoria):")
    display(X_codificado.head())
else:
    X_codificado = X.copy()
    print("\nNenhum atributo categórico encontrado — nenhuma codificação necessária.")

colunas_modelo = list(X_codificado.columns)

# Codificação da classe alvo (target). Aqui o Label Encoding é adequado, pois
# a classe é usada apenas como rótulo (0/1/2...), não como valor numérico
# usado em cálculos de distância/escala.
le_classe = LabelEncoder()
y_encoded = le_classe.fit_transform(y.astype(str))
print(f"\nCodificação da variável-alvo 'Classe': {dict(zip(le_classe.classes_, le_classe.transform(le_classe.classes_)))}")

# Padronização (StandardScaler): a Regressão Logística com regularização é sensível à
# escala das variáveis, então padronizamos todas as colunas (numéricas originais e
# dummies 0/1) para média 0 e desvio padrão 1 antes de treinar. Isso também torna os
# coeficientes comparáveis entre si.
scaler = StandardScaler()
X_escalado = scaler.fit_transform(X_codificado)
X_escalado = pd.DataFrame(X_escalado, columns=colunas_modelo, index=X_codificado.index)

print("\nTabela final após padronização (é isto que entra no modelo):")
display(X_escalado.head())

print("\nColunas de features usadas no modelo:", colunas_modelo)
print("Classes encontradas:", list(le_classe.classes_))


# ================================================
# 5. TREINAMENTO DO MODELO DE REGRESSÃO LOGÍSTICA
# ================================================
print("\n=== Treinando o modelo ===")


def criar_modelo():
    parametros = dict(
        penalty=penalty,
        C=C,
        class_weight=class_weight,
        fit_intercept=fit_intercept,
        max_iter=max_iter,
        solver=solver,
        random_state=random_state,
    )
    if l1_ratio is not None:
        parametros["l1_ratio"] = l1_ratio
    return LogisticRegression(**parametros)


if test_size and test_size > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X_escalado, y_encoded, test_size=test_size, random_state=random_state,
        stratify=y_encoded if len(set(y_encoded)) > 1 else None
    )

    modelo = criar_modelo()
    modelo.fit(X_train, y_train)

    y_pred_test = modelo.predict(X_test)
    acuracia = accuracy_score(y_test, y_pred_test)

    print(f"Acurácia na validação interna ({test_size*100:.0f}% dos dados): {acuracia*100:.2f}%\n")
    print("Relatório de classificação:")
    print(classification_report(y_test, y_pred_test, target_names=[str(c) for c in le_classe.classes_]))

    # Treina o modelo final com todos os dados disponíveis
    modelo_final = criar_modelo()
    modelo_final.fit(X_escalado, y_encoded)
else:
    modelo_final = criar_modelo()
    modelo_final.fit(X_escalado, y_encoded)
    print("Modelo treinado com 100% dos dados de treinamento (sem validação interna).")

print("Treinamento concluído.")

# Verifica se é um problema binário (1 linha de coeficientes) ou multiclasse (1 linha por classe)
n_classes = len(le_classe.classes_)
eh_binario = modelo_final.coef_.shape[0] == 1


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def softmax(vetor_z):
    exp_z = np.exp(vetor_z - np.max(vetor_z))
    return exp_z / exp_z.sum()


# =====================================================================
# 6. EQUAÇÃO(ÕES) z OBTIDA(S) PELO MODELO
# (Atenção: os coeficientes abaixo se referem às features PADRONIZADAS
#  pelo StandardScaler — incluindo as colunas dummy 0/1 do one-hot — e
#  não aos valores originais das colunas.)
# =====================================================================
print("\n" + "=" * 70)
print("EQUAÇÃO(ÕES) z DO MODELO (aplicadas sobre as features padronizadas)")
print("=" * 70)

if eh_binario:
    classe_positiva = le_classe.classes_[1]
    classe_negativa = le_classe.classes_[0]
    coefs = modelo_final.coef_[0]
    intercepto = modelo_final.intercept_[0]

    termos = " + ".join([f"({coef:+.4f} × {feat})" for coef, feat in zip(coefs, colunas_modelo)])
    print(f"\nz (log-odds para a classe '{classe_positiva}') =")
    print(f"  {intercepto:+.4f} + {termos}")
    print(f"\nProbabilidade da classe '{classe_positiva}' = 1 / (1 + e^(-z))")
    print(f"Probabilidade da classe '{classe_negativa}' = 1 - Probabilidade('{classe_positiva}')")
else:
    for i, classe in enumerate(le_classe.classes_):
        coefs = modelo_final.coef_[i]
        intercepto = modelo_final.intercept_[i]
        termos = " + ".join([f"({coef:+.4f} × {feat})" for coef, feat in zip(coefs, colunas_modelo)])
        print(f"\nz da classe '{classe}' =")
        print(f"  {intercepto:+.4f} + {termos}")
    print("\nProbabilidade de cada classe = softmax(z_classe1, z_classe2, ...)")


# =====================================================================
# 7. ODDS RATIO (SÓ PARA O CASO BINÁRIO) — leitura interessante extra
# =====================================================================
if eh_binario:
    odds_ratio = np.exp(coefs)
    tabela_odds = pd.DataFrame({
        "Feature": colunas_modelo,
        "Coeficiente": coefs.round(4),
        "Odds Ratio (e^coef)": odds_ratio.round(4),
        "Efeito ao aumentar a feature": [
            f"Aumenta a chance de '{classe_positiva}'" if c > 0 else f"Diminui a chance de '{classe_positiva}'"
            for c in coefs
        ]
    }).sort_values("Coeficiente", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

    print("\n=== Odds Ratio por feature (interpretação dos coeficientes) ===")
    print(f"(cada unidade de aumento na feature PADRONIZADA multiplica a chance de '{classe_positiva}' pelo Odds Ratio)")
    print("Para as colunas dummy (0/1) geradas pelo one-hot, isso equivale a comparar a categoria")
    print("indicada pela coluna contra a categoria de referência daquele atributo.")
    display(tabela_odds)


# ===================================================
# 8. CARREGAMENTO DO INDIVÍDUO A SER CLASSIFICADO
# ===================================================
print("\n=== Carregando dados de entrada ===")
caminho_entrada = localizar_planilha(PASTA_ENTRADA)
print(f"Arquivo de entrada encontrado: {caminho_entrada}")

df_entrada = pd.read_excel(caminho_entrada)

# Garante que a planilha de entrada tenha as mesmas colunas de atributos ORIGINAIS
# (antes do one-hot) usadas no treinamento
colunas_faltantes = [c for c in colunas_originais if c not in df_entrada.columns]
if colunas_faltantes:
    raise ValueError(f"A planilha de entrada não possui as colunas: {colunas_faltantes}")

X_novo = df_entrada[colunas_originais].copy()

# Aplica o mesmo tratamento de valores ausentes usado no treinamento
for col in colunas_numericas:
    X_novo[col] = X_novo[col].fillna(medianas_treino[col])
for col in colunas_categoricas:
    X_novo[col] = X_novo[col].fillna("Desconhecido")

# Aplica o mesmo One-Hot Encoding usado no treinamento
print("\n" + "=" * 70)
print("TRANSFORMAÇÃO DOS INDIVÍDUOS DE ENTRADA (mesmo One-Hot do treinamento)")
print("=" * 70)

if colunas_categoricas:
    print("\nTabela ANTES da transformação (valores originais dos indivíduos):")
    display(X_novo[colunas_categoricas])

    X_novo_codificado = pd.get_dummies(X_novo, columns=colunas_categoricas, drop_first=True)

    # Verifica se apareceu alguma categoria nova, nunca vista no treinamento
    colunas_novas_categorias = [c for c in X_novo_codificado.columns if c not in colunas_modelo and c not in colunas_numericas]
    if colunas_novas_categorias:
        print(f"\nAviso: categorias não vistas no treinamento foram encontradas e serão tratadas "
              f"como a categoria de referência (todas as colunas dummy = 0): {colunas_novas_categorias}")
else:
    X_novo_codificado = X_novo.copy()

# Garante que as colunas fiquem exatamente iguais às do treinamento, na mesma ordem
# (colunas dummy ausentes viram 0 = categoria de referência; colunas extras são descartadas)
X_novo_codificado = X_novo_codificado.reindex(columns=colunas_modelo, fill_value=0)

print("\nTabela DEPOIS da transformação (0/1 por categoria, já alinhada ao treinamento):")
display(X_novo_codificado)

# Aplica a mesma padronização (usando o scaler já ajustado no treinamento)
X_novo_escalado = scaler.transform(X_novo_codificado)
X_novo_escalado = pd.DataFrame(X_novo_escalado, columns=colunas_modelo, index=X_novo_codificado.index)

print(f"\n{len(X_novo)} indivíduo(s) carregado(s) para classificação (uma predição será feita para cada linha da planilha):")
tabela_individuos = df_entrada[colunas_originais].copy()
tabela_individuos.index = range(1, len(tabela_individuos) + 1)
tabela_individuos.index.name = "Indivíduo"
display(tabela_individuos)


# =========================================================
# 9. CLASSIFICAÇÃO E PROBABILIDADES DE CADA INDIVÍDUO
# (o laço abaixo percorre TODAS as linhas/indivíduos da planilha de ENTRADA,
#  calculando manualmente o valor de z, comparando com o predict_proba do
#  scikit-learn como conferência, e exibindo os valores de probabilidade
#  de cada classe — sem barras gráficas)
# =========================================================
for idx in range(len(X_novo_escalado)):
    linha_escalada = X_novo_escalado.iloc[idx].values

    print("=" * 60)
    print(f"Indivíduo {idx + 1} de {len(X_novo_escalado)}")
    print("=" * 60)
    print("\nDados do indivíduo:")
    dados_individuo = df_entrada.iloc[[idx]][colunas_originais].copy()
    dados_individuo.index = [idx + 1]
    dados_individuo.index.name = "Indivíduo"
    display(dados_individuo)

    if eh_binario:
        z_calculado = np.dot(coefs, linha_escalada) + intercepto
        prob_positiva = sigmoid(z_calculado)
        prob_negativa = 1 - prob_positiva
        probs_dict = {classe_negativa: prob_negativa, classe_positiva: prob_positiva}

        print(f"\nz calculado manualmente = {z_calculado:.4f}")
        print(f"(conferência via modelo: {modelo_final.decision_function(linha_escalada.reshape(1, -1))[0]:.4f})")
    else:
        z_vetor = np.array([
            np.dot(modelo_final.coef_[i], linha_escalada) + modelo_final.intercept_[i]
            for i in range(n_classes)
        ])
        probs_vetor = softmax(z_vetor)
        probs_dict = dict(zip(le_classe.classes_, probs_vetor))

        print("\nz calculado manualmente para cada classe:")
        for classe, z_c in zip(le_classe.classes_, z_vetor):
            print(f"  z('{classe}') = {z_c:.4f}")

    classe_predita = le_classe.inverse_transform(modelo_final.predict(linha_escalada.reshape(1, -1)))[0]

    print(f"\n>>> CLASSE PREDITA: {classe_predita} <<<\n")
    print("Probabilidades por classe:")
    for classe, prob in sorted(probs_dict.items(), key=lambda item: item[1], reverse=True):
        print(f"  {str(classe):>20s} : {prob*100:6.2f}%")

    resultado_df = pd.DataFrame({
        "Classe": list(probs_dict.keys()),
        "Probabilidade (%)": [round(p * 100, 2) for p in probs_dict.values()]
    }).sort_values("Probabilidade (%)", ascending=False).reset_index(drop=True)

    display(resultado_df)
    print("\n")


Bibliotecas importadas com sucesso.

=== Parâmetros da Regressão Logística ===



Número máximo de iterações para convergência (max_iter) [padrão = 1000]:  
Proporção dos dados que será separada para VALIDAÇÃO/TESTE, e não usada no treino (ex.: 0.2 = 20% para teste e 80% para treino; use 0 para treinar com 100% dos dados) [padrão = 0.2]:  0.2



=== Parâmetros definidos ===
penalty         = l2 (padrão)
solver          = lbfgs (escolhido automaticamente conforme o penalty)
C               = 1.0 (padrão)
class_weight    = None (padrão, sem tratamento de desbalanceamento)
fit_intercept   = True (padrão, intercepto incluído)
max_iter        = 1000
random_state    = 42
test_size       = 0.2

=== Carregando dados de treinamento ===
Arquivo de treinamento encontrado: C:\Users\vang_\anaconda3\Scripts\DATA SCIENCE MODELS\CLASSIFICAÇÃO\DADOS DE TREINAMENTO\Dados de Treinamento - prontos 2.xlsx
Dimensões dos dados: 30 linhas x 4 colunas


,Faixa de idades,Faixa de logs de renda,Faixa de tickets médios,Classe
0,Acima de 50,Média,Alto,1
1,Acima de 50,Média,Alto,1
2,Acima de 50,Baixa,Alto,1
3,Acima de 50,Alta,Alto,1
4,Acima de 50,Baixa,Médio,0



Colunas categóricas identificadas: ['Faixa de idades', 'Faixa de logs de renda', 'Faixa de tickets médios']
Colunas numéricas identificadas: (nenhuma)

TRANSFORMAÇÃO: ONE-HOT ENCODING DOS ATRIBUTOS CATEGÓRICOS

Tabela ANTES da transformação (valores originais):


...

,Classe,Probabilidade (%)
0,1,90.91
1,0,9.09




Indivíduo 11 de 11

Dados do indivíduo:


,Faixa de idades,Faixa de logs de renda,Faixa de tickets médios
Indivíduo,,,
11,Acima de 50,Alta,Médio



z calculado manualmente = 1.2126
(conferência via modelo: 1.2126)

>>> CLASSE PREDITA: 1 <<<

Probabilidades por classe:
                     1 :  77.08%
                     0 :  22.92%


,Classe,Probabilidade (%)
0,1,77.08
1,0,22.92
